# Монгол F5-TTS inference (Modal-гүйгээр, Colab дээр шууд)

Modal-ийн billing spend limit-д хүрсэн үед зориулсан **backup** — жинхэнэ production зам биш, харин demo/тест хийхэд ашиглана.

`gpu/f5_modal.py`-тай яг ижил логик: `use_ema=False`, батлагдсан Монгол reference клип, `fix_duration` (segment-ийн жинхэнэ уртаар audio-г тааруулна — F5-ийн UTF-8 байтын урт дээр суурилсан auto duration тооцооллыг тойрч гардаг, учир нь тэр нь Кирилл/Latin холилдоход буруу гардаг).

**Өмнө нь бэлдэх зүйлс (Google Drive-ийн MyDrive руу байршуул):**
- `mn_model_last.pt` (эсвэл жижигрүүлсэн inference-only checkpoint)
- `mn_vocab.txt`
- `ref_male_v2.wav` — батлагдсан reference клип (9.2 сек, 24kHz mono). Downloads-с чинь: `C:\Users\Huawei\Downloads\ref_male_v2.wav`

**Runtime:** дээд цэснээс `Runtime → Change runtime type → GPU` (T4-г ч хангалттай).

## 1. GPU шалгах

In [ ]:
!nvidia-smi

## 2. f5-tts суулгах (~1-2 мин)

In [ ]:
!pip install -q f5-tts soundfile

## 3. Google Drive холбож, файлын замууд

Файлууд өөр байршилд байвал доорх замуудыг зас.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_PATH = "/content/drive/MyDrive/mn_model_last.pt"
VOCAB_PATH = "/content/drive/MyDrive/mn_vocab.txt"
REF_AUDIO_PATH = "/content/drive/MyDrive/ref_male_v2.wav"

for p in (MODEL_PATH, VOCAB_PATH, REF_AUDIO_PATH):
    size_mb = os.path.getsize(p) / 1e6 if os.path.exists(p) else None
    print(f"{'OK ' if size_mb else 'MISSING'} {p}  ({size_mb:.1f} MB)" if size_mb else f"MISSING {p}")

## 4. Модел ачаалах

⚠️ **`use_ema=False` заавал!** EMA жин нь сургалт олон удаа зогсож эхэлсэн тул эрт үеийн (буруу дуудлагатай) checkpoint дээр царцсан. Жинхэнэ (зөв) моделийн жин нь `use_ema=False`-ээр ачаалагдана.

In [ ]:
from f5_tts.api import F5TTS

model = F5TTS(
    model="F5TTS_v1_Base",
    ckpt_file=MODEL_PATH,
    vocab_file=VOCAB_PATH,
    use_ema=False,
)
print("Model loaded.")

## 5. Reference хоолой (батлагдсан клип)

Энэ бол өнөөдөр туршиж баталгаажуулсан клип (`clone_test_v4_mn_fixdur.wav` гаргахад ашигласан). `REF_TEXT` нь тухайн клипийн ЯГ ЗӨВ транскрипт — Whisper автомат уншилт Монголыг андуурдаг тул гараар бичсэн.

In [ ]:
import soundfile as sf
from IPython.display import Audio, display

REF_TEXT = (
    "Сайн байцгаана уу, киночид оо. Өнөөдөр та бүхэнд мянга есөн зуун ерэн хоёр "
    "онд нээлтээ хийсэн гол дүрд нь Жеки Чан тоглосон супер поп киног ярьж "
    "өгөхөөр бэлдлээ."
)

ref_info = sf.info(REF_AUDIO_PATH)
REF_SECONDS = ref_info.duration
print(f"Reference: {REF_SECONDS:.1f}s, {ref_info.samplerate}Hz, {ref_info.channels}ch")
display(Audio(REF_AUDIO_PATH))

## 6. Synthesize функц

`fix_duration = ref_seconds + target_duration` — F5-ийн UTF-8 байтын урт дээр суурилсан auto duration тооцооллыг тойрч, segment-ийн жинхэнэ хугацаанд яг тааруулна (dashboard-ийн subtitle-той синхрон явахад чухал).

In [ ]:
def synthesize(gen_text: str, target_duration: float, out_path: str | None = None):
    fix_duration = REF_SECONDS + max(0.4, target_duration)
    wav, sr, _ = model.infer(
        ref_file=REF_AUDIO_PATH,
        ref_text=REF_TEXT,
        gen_text=gen_text,
        nfe_step=40,       # gpu/f5_modal.py-тай ижил тохиргоо
        cfg_strength=1.8,
        fix_duration=fix_duration,
    )
    if out_path:
        sf.write(out_path, wav, sr)
    return wav, sr

## 7. Demo сегментүүд

Доорх жагсаалтад маргаашийн demo-д хэрэгтэй Монгол өгүүлбэр (болон түүний зорилтот үргэлжлэх хугацаа секундээр) — dashboard дээрх орчуулгаас хуулж ир. Ажиллуулбал `/content/demo_out/` дотор WAV файл болгон хадгална, доор сонсож шалгана.

In [ ]:
import os
os.makedirs("/content/demo_out", exist_ok=True)

# (Монгол текст, зорилтот үргэлжлэх хугацаа сек) — өөрсдийн segment-үүдээрээ соль
DEMO_SEGMENTS = [
    ("Сайн байна уу, энэ бол миний хоолойгоор орчуулсан жишээ юм.", 3.0),
]

for i, (text, duration) in enumerate(DEMO_SEGMENTS):
    out_path = f"/content/demo_out/segment_{i:03d}.wav"
    wav, sr = synthesize(text, duration, out_path)
    print(f"[{i}] {duration}s target → {len(wav)/sr:.2f}s actual — {out_path}")
    display(Audio(out_path))

## 8. Drive рүү хадгалах / татаж авах

In [ ]:
!cp -r /content/demo_out /content/drive/MyDrive/demo_out
print("Хадгалагдлаа: /content/drive/MyDrive/demo_out")